In [42]:
import warnings
for warn in [UserWarning, FutureWarning]: 
    warnings.filterwarnings("ignore", category=warn)

import os
import sys
import urllib.request
import zipfile

# --- Deep Learning & Computer Vision ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.models as models
from torchvision import transforms

import PIL
from PIL import Image

# --- Data Science & Analytics ---
import numpy as np
import polars as pl
import pandas as pd

# --- UI & Visualization ---
import ipywidgets
import jupyterlab as jlab
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from mpl_toolkits.mplot3d import Axes3D
from tqdm.notebook import tqdm

# --- Настройка путей для VSSD ---
current_dir = os.getcwd()
vssd_path = os.path.join(current_dir, 'VSSD', 'classification')
if vssd_path not in sys.path:
    sys.path.append(vssd_path)

# Официальные фабрики авторов VSSD
from config import get_config
from models import build_model

In [43]:
# Версии необходимых библиотек
packages = [
    "Torch", "Torchvision", "Pillow", "NumPy", "Polars", "Pandas", "Matplotlib", "Ipywidgets", "JupyterLab"
]

package_objects = [
    torch, torchvision, PIL, np, pl, pd, mpl, ipywidgets, jlab
]

versions = list(map(lambda obj: obj.__version__, package_objects))

columns_order = ["№", "Библиотека", "Версия"]
df_pkgs = (
    pl.DataFrame({
        columns_order[1]: packages,
        columns_order[2]: versions
    })
    .with_columns(pl.arange(1, pl.lit(len(packages)) + 1).alias(columns_order[0]))
    .select(columns_order)
)

display(df_pkgs)

# Генерация requirements.txt
path2reqs = "."
reqs_name = "requirements.txt"

def get_packages_and_versions():
    """Генерация строк с библиотеками и их версиями в формате: библиотека==версия"""
    for package, version in zip(packages, versions):
        yield f"{package.lower()}=={version}\n"

with open(os.path.join(path2reqs, reqs_name), "w", encoding="utf-8") as f:
    f.writelines(get_packages_and_versions())
    
print(f"📄 Файл {reqs_name} успешно сгенерирован!")


№,Библиотека,Версия
i64,str,str
1,"""Torch""","""2.2.2+cu121"""
2,"""Torchvision""","""0.17.2+cu121"""
3,"""Pillow""","""12.0.0"""
4,"""NumPy""","""1.26.4"""
5,"""Polars""","""1.38.1"""
6,"""Pandas""","""2.3.3"""
7,"""Matplotlib""","""3.10.8"""
8,"""Ipywidgets""","""8.1.8"""
9,"""JupyterLab""","""4.5.5"""


📄 Файл requirements.txt успешно сгенерирован!


In [44]:
# --- Класс-заглушка для парсера конфигов авторов ---
class DummyArgs:
    def __init__(self, cfg_path):
        self.cfg = cfg_path
        self.opts = []
        self.output = "output"
        self.tag = "default"
    def __getattr__(self, item):
        return None

# --- Инициализация официальной модели ---
cfg_file = os.path.join('VSSD', 'classification', 'configs', 'vssd', 'vssd_tiny_e300.yaml')
args = DummyArgs(cfg_file)

print("Сборка официальной архитектуры VSSD-Tiny...")
config = get_config(args)
model = build_model(config)
model.cuda()
model.eval() # Сразу переводим в режим оценки

# --- Загрузка весов ---
checkpoint_path = "vssd_tiny_best_ema.pth"
checkpoint = torch.load(checkpoint_path, map_location='cuda')
state_dict = checkpoint['model']

if any(k.startswith('module.') for k in state_dict.keys()):
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

model.load_state_dict(state_dict, strict=False)
print("Оригинальные веса ImageNet загружены")

# --- Функция инференса (Пайплайн ImageNet) ---
transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict_image(image_path):
    """Классифицирует изображение и возвращает топ-1 класс."""
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).cuda()
    
    with torch.no_grad():
        out = model(tensor)
        
    probs = F.softmax(out, dim=1)
    top_prob, top_idx = torch.topk(probs, 1)
    return top_idx[0][0].item(), top_prob[0][0].item()

# Тестовый прогон
dog_id, dog_prob = predict_image("test_dog.jpg")
print(f"Тест на собаке: ID {dog_id} | Уверенность: {dog_prob*100:.2f}%")


Сборка официальной архитектуры VSSD-Tiny...
=> merge config from VSSD/classification/configs/vssd/vssd_tiny_e300.yaml
Оригинальные веса ImageNet загружены
Тест на собаке: ID 207 | Уверенность: 65.28%


In [45]:
# --- Скачивание датасета ---
data_dir = 'hymenoptera_data'
if not os.path.exists(data_dir):
    print("Скачиваем датасет муравьев и пчел...")
    url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
    urllib.request.urlretrieve(url, "hymenoptera_data.zip")
    with zipfile.ZipFile("hymenoptera_data.zip", 'r') as zip_ref:
        zip_ref.extractall(".")

# --- Настройка трансформаций (Augmentation для Train, Стандарт для Val) ---
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(), # Учим модель распознавать отзеркаленных пчел
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'val']}

dataloaders = {x: DataLoader(image_datasets[x], batch_size=16, shuffle=True, num_workers=4)
               for x in ['train', 'val']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"Данные загружены! Классы: {class_names}")
print(f"Тренировочных изображений: {dataset_sizes['train']} | Валидационных: {dataset_sizes['val']}")

# --- Замена "головы" VSSD ---
# Замораживаем основное "тело" модели, чтобы не сломать веса на первых эпохах
for param in model.parameters():
    param.requires_grad = False

# Меняем голову с 1000 классов (ImageNet) на 2 класса (Пчелы/Муравьи)
in_features = model.head.in_features
model.head = nn.Linear(in_features, len(class_names)).cuda()

print(f"Голова VSSD-Tiny успешно заменена! Модель готова предсказывать {len(class_names)} класса.")


Данные загружены! Классы: ['ants', 'bees']
Тренировочных изображений: 244 | Валидационных: 153
Голова VSSD-Tiny успешно заменена! Модель готова предсказывать 2 класса.


In [46]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.head.parameters(), lr=0.001)

def train_model(model, criterion, optimizer, num_epochs=5):
    since = time.time()
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Эпоха {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  
            else:
                model.eval()   

            running_loss = 0.0
            running_corrects = 0

            # Классический цикл без tqdm
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.cuda()
                labels = labels.cuda()

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double().item() / dataset_sizes[phase]

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Обучение завершено за {time_elapsed // 60:.0f}м {time_elapsed % 60:.0f}с')
    print(f'Лучшая точность на валидации: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model

model = train_model(model, criterion, optimizer, num_epochs=5)

Эпоха 1/5
----------
Train Loss: 0.5051 | Acc: 0.7705
Val Loss: 0.1509 | Acc: 1.0000

Эпоха 2/5
----------
Train Loss: 0.1600 | Acc: 0.9631
Val Loss: 0.0672 | Acc: 1.0000

Эпоха 3/5
----------
Train Loss: 0.1430 | Acc: 0.9549
Val Loss: 0.0462 | Acc: 1.0000

Эпоха 4/5
----------
Train Loss: 0.1130 | Acc: 0.9631
Val Loss: 0.0416 | Acc: 1.0000

Эпоха 5/5
----------
Train Loss: 0.0933 | Acc: 0.9713
Val Loss: 0.0377 | Acc: 1.0000

Обучение завершено за 0м 8с
Лучшая точность на валидации: 1.0000


In [39]:
save_path = "vssd_tiny_ants_bees.pth"
torch.save(model.state_dict(), save_path)
print(f"Веса модели сохранены в файл: {save_path}")

Веса модели сохранены в файл: vssd_tiny_ants_bees.pth


In [47]:
# Сравним с ResNet-18

# Используем актуальный синтаксис PyTorch для загрузки весов
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).cuda()

#  Замораживаем "тело"
for param in resnet.parameters():
    param.requires_grad = False

# Меняем "голову" (у ResNet это слой .fc)
num_ftrs = resnet.fc.in_features
resnet.fc = nn.Linear(num_ftrs, len(class_names)).cuda()
optimizer_resnet = optim.AdamW(resnet.fc.parameters(), lr=0.001)

resnet = train_model(resnet, criterion, optimizer_resnet, num_epochs=5)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/max/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:04<00:00, 11.4MB/s]

Эпоха 1/5
----------


Train Loss: 0.6674 | Acc: 0.5943
Val Loss: 0.5301 | Acc: 0.7190

Эпоха 2/5
----------
Train Loss: 0.4293 | Acc: 0.8320
Val Loss: 0.3237 | Acc: 0.8758

Эпоха 3/5
----------
Train Loss: 0.3176 | Acc: 0.9016
Val Loss: 0.2638 | Acc: 0.9020

Эпоха 4/5
----------
Train Loss: 0.2886 | Acc: 0.8893
Val Loss: 0.2743 | Acc: 0.9020

Эпоха 5/5
----------
Train Loss: 0.3047 | Acc: 0.8934
Val Loss: 0.2227 | Acc: 0.9150

Обучение завершено за 0м 4с
Лучшая точность на валидации: 0.9150


In [ ]:
'''
Вывод: VSSD справилась с задачей лучше ResNet, точность на валидационной выборке выше на 8.5%
'''